# PEMDT-Cirrhosis — Progression to Decompensation (survival redesign)

Rebuilt to fix the weak outcome and use the timing properly. Instead of predicting "first
cirrhosis ICD code" (a documentation event), this models **time to progression to
*decompensated* cirrhosis** — ascites, variceal bleeding, hepatic encephalopathy, SBP, or
hepatorenal syndrome — among **at-risk, non-cirrhotic** patients.

## What changed vs v1
- **Outcome:** time-to-**decompensation** (a concrete clinical event), not the cirrhosis code.
- **Richer observations:** added MELD physiology — Creatinine, INR, Sodium, Hemoglobin, WBC —
  on top of AST/ALT/Platelets/Albumin/Bilirubin (so the latent stage isn't just FIB-4).
- **Trajectory dynamics:** stage velocity, up-transitions, dwell time in the top stage, and
  stage-sequence entropy — "0→1→2" and "1→1→1" have the same peak but very different risk.
- **Survival model (not AUC):** a Cox proportional-hazards model with the latent stage as a
  covariate → **hazard ratios** and **concordance (C-index)**, benchmarked against FIB-4 and
  MELD-Na, plus **1-year decompensation risk by baseline stage**.
- The HMM stays **unsupervised** (glass-box); the outcome enters only through a **hazard layer
  on top**, so the staging is not contaminated by the label.

## Honest caveats
- MIMIC has short, sparse, irregular follow-up, so survival times are **heavily censored** and
  short — hazard estimates are noisier than a registry with real longitudinal follow-up.
- "Decompensation" is still ICD/marker-based, but far more concrete than a cirrhosis code.
- **Baseline-anchored** Cox (covariates from the first visit) is the clean model; the
  trajectory-augmented model peeks across pre-event visits, so it carries a mild
  immortal-time caveat — reported separately and flagged.

*Requires MIMIC-IV `hosp/`: diagnoses_icd, admissions, patients, labevents. Needs `lifelines`.*

## HMM library

In [1]:
"""Missing-aware diagonal-Gaussian HMM with optional treatment fusion.
Modes:
  baseline   : Gaussian emissions, homogeneous transitions
  emission   : Gaussian + Bernoulli(treatment) emissions, homogeneous transitions
  transition : Gaussian emissions, treatment-conditioned transitions (A_off / A_on)
"""
import numpy as np

VAR_FLOOR = 1e-3

def emission_logB(O, mu, var, u=None, p_treat=None):
    """O: T x D (NaN allowed). returns logB: T x K. Optional Bernoulli treatment emission."""
    T, D = O.shape; K = mu.shape[0]
    mask = ~np.isnan(O)                                  # T x D
    diff = O[:, None, :] - mu[None, :, :]                # T x K x D
    term = -0.5 * (np.log(2*np.pi*var)[None] + (diff**2)/var[None])
    term = np.where(mask[:, None, :], term, 0.0)
    logB = term.sum(axis=2)                              # T x K
    if u is not None and p_treat is not None:
        pt = np.clip(p_treat, 1e-4, 1-1e-4)              # K
        logB = logB + (u[:, None]*np.log(pt)[None] + (1-u[:, None])*np.log(1-pt)[None])
    return logB

def fwd_bwd(logB, pi, A_t):
    """A_t: (T-1) x K x K time-indexed transition matrices. Scaled forward-backward."""
    T, K = logB.shape
    m = logB.max(axis=1); B = np.exp(logB - m[:, None])
    alpha = np.zeros((T, K)); c = np.zeros(T)
    alpha[0] = pi * B[0]; c[0] = alpha[0].sum() + 1e-300; alpha[0] /= c[0]
    for t in range(1, T):
        alpha[t] = (alpha[t-1] @ A_t[t-1]) * B[t]
        c[t] = alpha[t].sum() + 1e-300; alpha[t] /= c[t]
    beta = np.zeros((T, K)); beta[-1] = 1.0
    for t in range(T-2, -1, -1):
        beta[t] = (A_t[t] @ (B[t+1]*beta[t+1])) / c[t+1]
    gamma = alpha*beta; gamma /= gamma.sum(axis=1, keepdims=True) + 1e-300
    loglik = np.sum(np.log(c)) + np.sum(m)
    return alpha, beta, gamma, B, c, loglik

def _init(seqs, K, seed=0):
    rng = np.random.default_rng(seed)
    allobs = np.vstack([s["O"] for s in seqs])
    D = allobs.shape[1]
    med = np.nanmedian(allobs, axis=0)
    # severity proxy = mean of standardized obs (already standardized upstream) -> quantile init
    sev = np.nanmean(allobs, axis=1)
    qs = np.nanquantile(sev, np.linspace(0.1, 0.9, K))
    mu = np.tile(med, (K, 1)).astype(float)
    # spread state means along each dim by quantiles of that dim
    for d in range(D):
        col = allobs[:, d]; col = col[~np.isnan(col)]
        if len(col):
            mu[:, d] = np.quantile(col, np.linspace(0.2, 0.8, K))
    var = np.tile(np.nanvar(allobs, axis=0), (K, 1)); var[var < VAR_FLOOR] = 1.0
    pi = np.full(K, 1.0/K)
    A = np.full((K, K), 0.1/(K-1)) + np.eye(K)*0.9
    A /= A.sum(axis=1, keepdims=True)
    return pi, A, mu, var

def fit(seqs, K=4, mode="baseline", n_iter=25, seed=0, verbose=False):
    """seqs: list of dict with 'O' (T x D) and optional 'u' (T,) treatment binary."""
    pi, A, mu, var = _init(seqs, K, seed)
    A_on = A.copy(); A_off = A.copy()
    p_treat = np.linspace(0.2, 0.8, K) if mode == "emission" else None
    D = mu.shape[1]
    prev_ll = -np.inf
    for it in range(n_iter):
        pi_acc = np.zeros(K)
        num_mu = np.zeros((K, D)); den_mu = np.zeros((K, D)); num_var = np.zeros((K, D))
        if mode == "transition":
            xi_off = np.zeros((K, K)); xi_on = np.zeros((K, K))
        else:
            xi_sum = np.zeros((K, K))
        tp_num = np.zeros(K); tp_den = np.zeros(K)
        total_ll = 0.0
        for s in seqs:
            O = s["O"]; T = O.shape[0]
            u = s.get("u") if mode != "baseline" else None
            logB = emission_logB(O, mu, var, u=(u if mode == "emission" else None),
                                 p_treat=(p_treat if mode == "emission" else None))
            if mode == "transition" and u is not None:
                A_t = np.stack([A_on if u[t] > 0.5 else A_off for t in range(T-1)]) if T > 1 \
                      else np.zeros((0, K, K))
            else:
                A_t = np.stack([A]*(T-1)) if T > 1 else np.zeros((0, K, K))
            alpha, beta, gamma, B, c, ll = fwd_bwd(logB, pi, A_t)
            total_ll += ll
            pi_acc += gamma[0]
            mask = ~np.isnan(O)
            # mean AND variance from the SAME posteriors (cached gamma)
            Osq = np.where(mask, O*O, 0.0)
            for k in range(K):
                g = gamma[:, k]
                num_mu[k]  += np.nansum(np.where(mask, O, 0.0) * g[:, None], axis=0)
                den_mu[k]  += (mask * g[:, None]).sum(axis=0)
                num_var[k] += (Osq * g[:, None]).sum(axis=0)   # sum gamma*x^2 (suff. stat)
            for t in range(T-1):
                At = (A_on if (mode == "transition" and u is not None and u[t] > 0.5)
                      else A_off if mode == "transition" else A)
                xi = (alpha[t][:, None] * At * (B[t+1]*beta[t+1])[None, :]) / (c[t+1])
                xi /= xi.sum() + 1e-300
                if mode == "transition" and u is not None:
                    if u[t] > 0.5: xi_on += xi
                    else: xi_off += xi
                else:
                    xi_sum += xi
            if mode == "emission" and u is not None:
                tp_num += (gamma * u[:, None]).sum(axis=0)
                tp_den += gamma.sum(axis=0)
        # M-step
        N = len(seqs)
        pi = pi_acc / N; pi /= pi.sum()
        mu = num_mu / np.maximum(den_mu, 1e-6)
        var = num_var / np.maximum(den_mu, 1e-6) - mu**2   # E[x^2]-E[x]^2
        var[var < VAR_FLOOR] = VAR_FLOOR
        if mode == "transition":
            A_off = xi_off / np.maximum(xi_off.sum(axis=1, keepdims=True), 1e-12)
            A_on  = xi_on  / np.maximum(xi_on.sum(axis=1, keepdims=True), 1e-12)
        else:
            A = xi_sum / np.maximum(xi_sum.sum(axis=1, keepdims=True), 1e-12)
        if mode == "emission":
            p_treat = np.clip(tp_num/np.maximum(tp_den, 1e-6), 1e-4, 1-1e-4)
        if verbose:
            print(f"  iter {it+1}: loglik={total_ll:.1f}")
        if abs(total_ll - prev_ll) < 1e-3*abs(prev_ll+1e-9):
            break
        prev_ll = total_ll
    params = dict(pi=pi, mu=mu, var=var, K=K, mode=mode)
    if mode == "transition": params["A_off"], params["A_on"] = A_off, A_on
    else: params["A"] = A
    if mode == "emission": params["p_treat"] = p_treat
    params["loglik"] = total_ll
    return params

def reorder_by_severity(params, direction):
    """direction: array length D, sign of severity per obs dim. Sort states ascending severity."""
    mu = params["mu"]; sev = (mu * direction[None, :]).sum(axis=1)
    order = np.argsort(sev)
    inv = np.argsort(order)
    p = dict(params)
    p["mu"] = mu[order]; p["var"] = params["var"][order]; p["pi"] = params["pi"][order]
    p["pi"] /= p["pi"].sum()
    if params["mode"] == "transition":
        p["A_off"] = params["A_off"][np.ix_(order, order)]
        p["A_on"]  = params["A_on"][np.ix_(order, order)]
    else:
        p["A"] = params["A"][np.ix_(order, order)]
    if "p_treat" in params: p["p_treat"] = params["p_treat"][order]
    p["_order"] = order
    return p

def posterior_stage(params, O, u=None):
    """Return gamma (T x K) and expected stage per window."""
    mode = params["mode"]; T = O.shape[0]
    logB = emission_logB(O, params["mu"], params["var"],
                         u=(u if mode == "emission" else None),
                         p_treat=(params.get("p_treat") if mode == "emission" else None))
    if mode == "transition" and u is not None:
        A_t = np.stack([params["A_on"] if u[t] > 0.5 else params["A_off"] for t in range(T-1)]) if T>1 else np.zeros((0,params["K"],params["K"]))
    else:
        A = params.get("A", params.get("A_off"))
        A_t = np.stack([A]*(T-1)) if T>1 else np.zeros((0,params["K"],params["K"]))
    _,_,gamma,_,_,ll = fwd_bwd(logB, params["pi"], A_t)
    stages = (gamma * np.arange(params["K"])[None, :]).sum(axis=1)
    return gamma, stages, ll

def obs_pred_loglik(params, seqs):
    """FAIR cross-mode metric: one-step-ahead predictive log-lik of the next
    observation vector, P(O_{t+1} | O_{1:t}, u_{1:t}). Gaussian (lab) part only,
    so emission/transition/baseline are scored on the SAME target."""
    mode = params["mode"]; K = params["K"]; mu = params["mu"]; var = params["var"]
    A_def = params.get("A", params.get("A_off"))
    tot = 0.0; nwin = 0
    for s in seqs:
        O = s["O"]; T = O.shape[0]; u = s.get("u")
        logB = emission_logB(O, mu, var,
                             u=(u if mode == "emission" else None),
                             p_treat=(params.get("p_treat") if mode == "emission" else None))
        m = logB.max(axis=1); B = np.exp(logB - m[:, None])      # full emission for filtering
        gaussB = np.exp(emission_logB(O, mu, var) -              # gaussian-only for prediction
                        emission_logB(O, mu, var).max(axis=1)[:, None])
        gm = emission_logB(O, mu, var).max(axis=1)
        alpha = params["pi"] * B[0]; alpha /= alpha.sum() + 1e-300
        for t in range(T-1):
            if mode == "transition" and u is not None:
                A = params["A_on"] if u[t] > 0.5 else params["A_off"]
            else:
                A = A_def
            pred = alpha @ A
            pdens = (pred * gaussB[t+1]).sum()                   # predictive density (scaled)
            tot += np.log(pdens + 1e-300) + gm[t+1]              # undo gauss scaling
            nwin += 1
            alpha = pred * B[t+1]; alpha /= alpha.sum() + 1e-300
    return tot, nwin, tot/max(nwin, 1)

def heldout_loglik(params, seqs):
    tot = 0.0; nwin = 0
    for s in seqs:
        u = s.get("u")
        _,_,ll = posterior_stage(params, s["O"], u=u)
        tot += ll; nwin += s["O"].shape[0]
    return tot, nwin, tot/max(nwin,1)


## Config

In [2]:
# ============================================================
# CONFIG
# ============================================================
import os, json, warnings, time
import numpy as np
import pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore")

CANDIDATE_ROOTS = ["/Users/sumiyasayeed/Documents/mimic4/mimiciv",
                   "../data/mimiciv", "./data/mimiciv", "../../data/mimiciv"]
MIMIC_ROOT = next((p for p in CANDIDATE_ROOTS if os.path.exists(os.path.join(p,"hosp"))),
                  CANDIDATE_ROOTS[0])
HOSP = os.path.join(MIMIC_ROOT, "hosp")
OUT = Path("results/progression_output"); OUT.mkdir(exist_ok=True)
CHUNK = 1_000_000
K_STAGE = 3
MIN_VISITS = 2
RANDOM_SEED = 1
print("MIMIC_ROOT =", MIMIC_ROOT, "| hosp exists:", os.path.exists(HOSP))

def _csv(d,n):
    for e in (".csv",".csv.gz"):
        if os.path.exists(os.path.join(d,n+e)): return os.path.join(d,n+e)
    return os.path.join(d,n+".csv")

RISK = {
  "HCV":            {9:("07044","07054","07070","07071"), 10:("B182","B1920","B1921")},
  "HBV":            {9:("07022","07023","07032","07033","07042","07052"), 10:("B180","B181","B1910","B1911")},
  "alcohol":        {9:("303","3050","5710","5711","5713"), 10:("F10","K700","K701","K709")},
  "NAFLD_NASH":     {9:("5718","5719"), 10:("K7581","K760","K7589")},
  "hemochromatosis":{9:("27501","27503"), 10:("E8310","E8311")},
}
CIRR = {9:("5712","5715","5716"),
        10:("K7030","K7031","K717","K740","K741","K742","K743","K744","K745","K746")}
# Decompensation = the concrete progression event (ascites/varices bleed/HE/SBP/HRS)
DECOMP = {9:("78959","7895","4560","45620","5722","56723","5724"),
          10:("R180","R188","R189","I8501","I8511","K720","K721","K729","K7290","K7291","K652","K767")}

LAB_IDS = {"AST":50878,"ALT":50861,"Platelets":51265,"Albumin":50862,"Bilirubin":50885,
           "Creatinine":50912,"INR":51237,"Sodium":50983,"Hemoglobin":51222,"WBC":51301}
OBS = list(LAB_IDS)                                   # 10-lab observation panel
LOG_OBS = {"AST","ALT","Bilirubin","Creatinine","WBC"}
SEV_DIR = {"AST":+1,"ALT":+1,"Platelets":-1,"Albumin":-1,"Bilirubin":+1,
           "Creatinine":+1,"INR":+1,"Sodium":-1,"Hemoglobin":-1,"WBC":+1}
print("Config loaded. obs =", OBS)


MIMIC_ROOT = /Users/sumiyasayeed/Documents/mimic4/mimiciv | hosp exists: True
Config loaded. obs = ['AST', 'ALT', 'Platelets', 'Albumin', 'Bilirubin', 'Creatinine', 'INR', 'Sodium', 'Hemoglobin', 'WBC']


## Step 1 — cohort + time-to-decompensation + labs

In [3]:
# ============================================================
# STEP 1 — at-risk cohort, time-to-DECOMPENSATION, per-visit labs
# ============================================================
def _pref(code_up, ver, p9, p10):
    return ((ver==9)&code_up.str.startswith(p9)) | ((ver==10)&code_up.str.startswith(p10))

dx = pd.read_csv(_csv(HOSP,"diagnoses_icd"), usecols=["subject_id","hadm_id","icd_code","icd_version"])
code = dx["icd_code"].astype(str).str.replace(".","",regex=False).str.upper()
dx["is_cirr"]   = _pref(code, dx["icd_version"], CIRR[9], CIRR[10]).astype(int)
dx["is_decomp"] = _pref(code, dx["icd_version"], DECOMP[9], DECOMP[10]).astype(int)
for n,c in RISK.items():
    dx[n] = _pref(code, dx["icd_version"], c[9], c[10]).astype(int)
perh = dx.groupby(["subject_id","hadm_id"])[["is_cirr","is_decomp"]+list(RISK)].max().reset_index()

adm = pd.read_csv(_csv(HOSP,"admissions"), usecols=["subject_id","hadm_id","admittime"])
adm["admittime"] = pd.to_datetime(adm["admittime"], errors="coerce")
adm = adm.merge(perh, on=["subject_id","hadm_id"], how="left").fillna(0).sort_values(["subject_id","admittime"])

base = adm.drop_duplicates("subject_id", keep="first").copy()
base["any_risk"] = base[list(RISK)].max(axis=1)
# at-risk AND not cirrhotic AND not already decompensated at baseline
cohort_subj = set(base[(base.any_risk==1) & (base.is_cirr==0) & (base.is_decomp==0)].subject_id)
adm_c = adm[adm.subject_id.isin(cohort_subj)].copy()
base_time = dict(zip(base.subject_id, base.admittime)); adm_c["base_time"] = adm_c.subject_id.map(base_time)

# event = first decompensation admission after baseline; else censor at last admission
post = adm_c[adm_c.admittime > adm_c.base_time]
first_dec = post[post.is_decomp==1].groupby("subject_id")["admittime"].min()
last_adm  = adm_c.groupby("subject_id")["admittime"].max()
surv = pd.DataFrame({"subject_id":list(cohort_subj)})
surv["base_time"] = surv.subject_id.map(base_time)
surv["dec_time"]  = surv.subject_id.map(first_dec)
surv["last_time"] = surv.subject_id.map(last_adm)
surv["event"] = surv["dec_time"].notna().astype(int)
end = surv["dec_time"].where(surv["event"]==1, surv["last_time"])
surv["duration_days"] = (end - surv["base_time"]).dt.total_seconds()/86400
surv = surv[surv["duration_days"] > 0]

# per-visit labs (pre-event)
hadm_set = set(adm_c.hadm_id.astype(int)); want = set(LAB_IDS.values())
acc = {}
for ch in pd.read_csv(_csv(HOSP,"labevents"), usecols=["hadm_id","itemid","valuenum"], chunksize=CHUNK, low_memory=False):
    ch = ch[ch.hadm_id.isin(hadm_set) & ch.itemid.isin(want) & ch.valuenum.notna()]
    if ch.empty: continue
    for (h,i),v in ch.groupby(["hadm_id","itemid"])["valuenum"].median().items():
        acc[(int(h),int(i))] = v
id2name = {v:k for k,v in LAB_IDS.items()}
labw = pd.DataFrame([{"hadm_id":h, **{id2name[i]:acc.get((h,i),np.nan) for i in want}} for h in hadm_set])
pts = pd.read_csv(_csv(HOSP,"patients"), usecols=["subject_id","anchor_age"])
visits = adm_c.merge(labw, on="hadm_id", how="left").merge(pts, on="subject_id", how="left")
visits = visits.merge(surv[["subject_id","dec_time","event","duration_days"]], on="subject_id", how="left")
visits = visits[visits.subject_id.isin(set(surv.subject_id))]
# censor visits at the decompensation event
keep = visits["dec_time"].isna() | (visits["admittime"] < visits["dec_time"])
visits = visits[keep].copy()
visits["FIB4"] = ((visits["anchor_age"]*visits["AST"]) /
                  (visits["Platelets"].clip(lower=1)*np.sqrt(visits["ALT"].clip(lower=1)))).clip(0,100)
# MELD-Na
b = visits["Bilirubin"].clip(lower=1); i_ = visits["INR"].clip(lower=1); c_ = visits["Creatinine"].clip(1,4)
meld = (3.78*np.log(b)+11.2*np.log(i_)+9.57*np.log(c_)+6.43).clip(6,40)
na = visits["Sodium"].clip(125,137)
visits["MELD_Na"] = (meld + 1.32*(137-na) - 0.033*meld*(137-na)).clip(6,40)

visits["has_obs"] = visits[OBS].notna().any(axis=1)
visits = visits[visits["has_obs"]].sort_values(["subject_id","admittime"])
vc = visits.groupby("subject_id").size(); serial = set(vc[vc>=MIN_VISITS].index)
visits = visits[visits.subject_id.isin(serial)]
surv = surv[surv.subject_id.isin(serial)]
print("="*60)
print(f"at-risk non-cirrhotic, not decompensated at baseline : {len(cohort_subj):,}")
print(f"  with >= {MIN_VISITS} pre-event lab visits ............ {visits.subject_id.nunique():,}")
print(f"  progress to DECOMPENSATION (events) ............ {int(surv.event.sum()):,} "
      f"({surv.event.mean()*100:.1f}%)")
print(f"  median follow-up (days) ........................ {surv.duration_days.median():.0f}")
print("="*60)


at-risk non-cirrhotic, not decompensated at baseline : 20,402
  with >= 2 pre-event lab visits ............ 5,597
  progress to DECOMPENSATION (events) ............ 301 (5.4%)
  median follow-up (days) ........................ 773


## Step 2 — HMM (richer obs) + trajectory dynamics

In [4]:
# ============================================================
# STEP 2 — unsupervised HMM (richer obs) + trajectory dynamics
# ============================================================
v = visits.copy()
for c in OBS:
    if c in LOG_OBS: v[c] = np.log1p(v[c].clip(lower=0))
STD = {}
for c in OBS:
    x = v[c].dropna().values
    if len(x) < 10: STD[c]=(0.0,1.0); continue
    med=float(np.median(x)); iqr=float(np.percentile(x,75)-np.percentile(x,25))
    sd=iqr/1.349 if iqr>1e-6 else float(np.std(x)); sd=sd if sd>1e-6 else 1.0
    STD[c]=(med,sd); v[c]=np.clip((v[c]-med)/sd,-8,8)

seqs=[]
for sid,g in v.groupby("subject_id"):
    O=g[OBS].values.astype(float)
    if len(O)<MIN_VISITS: continue
    seqs.append(dict(pid=sid,O=O,fib4=g["FIB4"].values,meld=g["MELD_Na"].values,
                     admit=pd.to_datetime(g["admittime"]).values))
t0=time.time()
params=fit(seqs,K=K_STAGE,mode="baseline",n_iter=30,seed=RANDOM_SEED)
sev=params["mu"] @ np.array([SEV_DIR[f] for f in OBS])
order=np.argsort(sev)
params=dict(params, mu=params["mu"][order], var=params["var"][order],
            pi=params["pi"][order]/params["pi"][order].sum(), A=params["A"][np.ix_(order,order)])
print(f"✅ HMM fit ({len(seqs):,} patients, {time.time()-t0:.1f}s); stage severity (asc):", np.round(sev[order],2))

def entropy(p):
    p=p[p>0]; return float(-(p*np.log(p)).sum()) if len(p) else 0.0
rows=[]
for s in seqs:
    g,st,_=posterior_stage(params,s["O"]); hard=np.argmax(g,axis=1)
    days=(pd.to_datetime(s["admit"]).astype("int64")/1e9/86400); span=max(np.ptp(days),1e-6)
    occ=np.bincount(hard, minlength=K_STAGE)/len(hard)
    rows.append(dict(pid=s["pid"],
        base_stage=float(st[0]), peak_stage=float(st.max()), last_stage=float(st[-1]), mean_stage=float(st.mean()),
        stage_velocity=float((st[-1]-st[0])/span*365),               # stages per year
        n_up=int((np.diff(hard)>0).sum()), dwell_top=float((hard==K_STAGE-1).mean()),
        stage_entropy=entropy(occ), n_visits=len(st),
        fib4_base=float(s["fib4"][0]) if np.isfinite(s["fib4"][0]) else np.nan,
        fib4_max=float(np.nanmax(s["fib4"])) if np.isfinite(s["fib4"]).any() else np.nan,
        meld_base=float(s["meld"][0]) if np.isfinite(s["meld"][0]) else np.nan,
        meld_max=float(np.nanmax(s["meld"])) if np.isfinite(s["meld"]).any() else np.nan))
SF=pd.DataFrame(rows).merge(surv[["subject_id","event","duration_days"]].rename(columns={"subject_id":"pid"}), on="pid")
print("\nDecompensation rate by baseline latent stage:")
print(SF.groupby(SF.base_stage.round()).agg(n=("pid","size"), event_rate=("event","mean"),
        median_FU=("duration_days","median")).round(3).to_string())


✅ HMM fit (5,597 patients, 3.0s); stage severity (asc): [-1.12  2.49  7.63]

Decompensation rate by baseline latent stage:
               n  event_rate  median_FU
base_stage                             
0.0         3352       0.033    919.608
1.0         1305       0.076    586.776
2.0          940       0.099    576.868


In [5]:
# ============================================================
# STEP 2b — grouped held-out patient split (parity with acute tasks)
# Fit the latent stage on TRAIN patients only; evaluate on TEST patients.
# ============================================================
rng = np.random.default_rng(RANDOM_SEED)
_pids = np.array([s["pid"] for s in seqs])
_perm = rng.permutation(len(_pids))
_n_test = int(0.30 * len(_pids))
_test_ids = set(_pids[_perm[:_n_test]])
train_seqs = [s for s in seqs if s["pid"] not in _test_ids]
test_seqs  = [s for s in seqs if s["pid"] in _test_ids]

# refit the HMM on train only, then re-order stages by severity (mirrors cell 8)
params_tr = fit(train_seqs, K=K_STAGE, mode="baseline", n_iter=30, seed=RANDOM_SEED)
_sev = params_tr["mu"] @ np.array([SEV_DIR[f] for f in OBS])
_o = np.argsort(_sev)
params_tr = dict(params_tr, mu=params_tr["mu"][_o], var=params_tr["var"][_o],
                 pi=params_tr["pi"][_o]/params_tr["pi"][_o].sum(),
                 A=params_tr["A"][np.ix_(_o, _o)])

def stage_features(P, S):
    rows = []
    for s in S:
        g, st, _ = posterior_stage(P, s["O"]); hard = np.argmax(g, axis=1)
        days = (pd.to_datetime(s["admit"]).astype("int64")/1e9/86400)
        span = max(np.ptp(days), 1e-6); occ = np.bincount(hard, minlength=K_STAGE)/len(hard)
        rows.append(dict(pid=s["pid"], base_stage=float(st[0]), peak_stage=float(st.max()),
            last_stage=float(st[-1]), mean_stage=float(st.mean()),
            stage_velocity=float((st[-1]-st[0])/span*365),
            n_up=int((np.diff(hard) > 0).sum()), dwell_top=float((hard == K_STAGE-1).mean()),
            stage_entropy=entropy(occ), n_visits=len(st),
            fib4_base=float(s["fib4"][0]) if np.isfinite(s["fib4"][0]) else np.nan,
            fib4_max=float(np.nanmax(s["fib4"])) if np.isfinite(s["fib4"]).any() else np.nan,
            meld_base=float(s["meld"][0]) if np.isfinite(s["meld"][0]) else np.nan,
            meld_max=float(np.nanmax(s["meld"])) if np.isfinite(s["meld"]).any() else np.nan))
    return pd.DataFrame(rows).merge(
        surv[["subject_id", "event", "duration_days"]].rename(columns={"subject_id": "pid"}), on="pid")

SF_train = stage_features(params_tr, train_seqs)
SF_test  = stage_features(params_tr, test_seqs)
print(f"grouped split: train={len(SF_train):,}  test={len(SF_test):,} patients "
      f"(HMM fit on train only)")
# To report HELD-OUT numbers, re-run STEP 3 (cell 11) with:  SF = SF_test

grouped split: train=3,918  test=1,679 patients (HMM fit on train only)


In [6]:
%pip install lifelines -q

Note: you may need to restart the kernel to use updated packages.


## Step 3 — survival model (Cox + 1-yr risk by stage)

In [7]:
# ============================================================
# STEP 3 — SURVIVAL: 1-yr risk by stage + 1-yr discrimination
# (dependency-free; uses lifelines for full Cox HR/C-index if available)
# ============================================================
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_predict
try:
    from lifelines import CoxPHFitter
    from lifelines.utils import concordance_index
    HAVE_LL = True
except Exception:
    HAVE_LL = False; print("lifelines not installed -> using dependency-free survival fallback\n")

D = SF_test.copy(); D["dur"] = D["duration_days"].clip(lower=1)

# ---- manual Kaplan-Meier 1-year risk by baseline stage ----
def km_risk(dur, ev, t=365):
    dur = np.asarray(dur, float); ev = np.asarray(ev, int)
    surv = 1.0
    for ut in np.unique(dur[ev == 1]):
        if ut > t: break
        d = int(((dur == ut) & (ev == 1)).sum()); r = int((dur >= ut).sum())
        if r > 0: surv *= (1 - d/r)
    return 1 - surv
print("=== 1-year decompensation risk by baseline latent stage (Kaplan-Meier) ===")
for k in range(K_STAGE):
    m = D["base_stage"].round() == k
    if m.sum() < 20: continue
    print(f"  baseline stage {k}: n={int(m.sum()):<5} 1-year risk = {km_risk(D.loc[m,'dur'], D.loc[m,'event'])*100:.1f}%")

# ---- 1-year discrimination benchmark (cases<=365d vs event-free>=365d) ----
lab = np.where((D.event==1)&(D.dur<=365), 1, np.where(D.dur>=365, 0, -1))
sel = lab >= 0; y = lab[sel]; Dd = D[sel].copy()
print(f"\n=== 1-year decompensation discrimination (n={int(sel.sum())}, events={int(y.sum())}) ===")
_DISC = (len(np.unique(y)) > 1 and y.sum() >= 10)
if not _DISC:
    print("  too few within-1-year events to benchmark discrimination (skipped)")
def boot(yt,s,n=1000,seed=42):
    rng=np.random.default_rng(seed); a=[]; yt=np.asarray(yt); s=np.asarray(s)
    for _ in range(n):
        idx=rng.integers(0,len(yt),len(yt))
        if not (1<yt[idx].sum()<len(idx)-1): continue
        a.append(roc_auc_score(yt[idx],s[idx]))
    return (np.median(a),np.percentile(a,2.5),np.percentile(a,97.5)) if a else (np.nan,)*3
for name,col in [("MELD-Na max","meld_max"),("FIB-4 max","fib4_max"),
                 ("Peak stage","peak_stage"),("Stage velocity","stage_velocity"),("Dwell top","dwell_top")]:
    if not _DISC or col not in Dd.columns: continue
    m=Dd[col].notna().values
    if m.sum()<20 or len(np.unique(y[m]))<2: continue
    md,lo,hi=boot(y[m],Dd[col].values[m]); print(f"  {name:<16} AUC={md:.3f} [{lo:.3f}-{hi:.3f}]")
def cv_auc(cols):
    cc=[c for c in cols if c in Dd.columns]
    X=StandardScaler().fit_transform(SimpleImputer(strategy='median').fit_transform(Dd[cc]))
    p=cross_val_predict(LogisticRegression(max_iter=2000,class_weight='balanced'),X,y,cv=5,method='predict_proba')[:,1]
    return roc_auc_score(y,p)
sfeat=["peak_stage","stage_velocity","dwell_top","stage_entropy"]
if _DISC:
    print(f"  {'Stage (combined)':<16} AUC={cv_auc(sfeat):.3f}")
    print(f"  {'Stage+MELD+FIB-4':<16} AUC={cv_auc(sfeat+['meld_max','fib4_max']):.3f}")

    # ---- paired bootstrap: AUC difference on the SAME patients ----
    from sklearn.model_selection import cross_val_predict as _cvp
    def _cv_scores(cols):
        cc=[c for c in cols if c in Dd.columns]
        Xs=StandardScaler().fit_transform(SimpleImputer(strategy='median').fit_transform(Dd[cc]))
        return _cvp(LogisticRegression(max_iter=2000,class_weight='balanced'),Xs,y,cv=5,method='predict_proba')[:,1]
    p_stage = _cv_scores(sfeat)
    p_meld  = Dd["meld_max"].values.astype(float)
    p_fib   = Dd["fib4_max"].values.astype(float)
    def _paired_delta(yt, s_better, s_ref, n=2000, seed=42):
        m = ~(np.isnan(s_better)|np.isnan(s_ref)); yt2=np.asarray(yt)[m]; a=np.asarray(s_better)[m]; b=np.asarray(s_ref)[m]
        rng=np.random.default_rng(seed); d=[]
        for _ in range(n):
            idx=rng.integers(0,len(yt2),len(yt2))
            if not (1<yt2[idx].sum()<len(idx)-1): continue
            d.append(roc_auc_score(yt2[idx],a[idx])-roc_auc_score(yt2[idx],b[idx]))
        return (np.median(d),np.percentile(d,2.5),np.percentile(d,97.5)) if d else (np.nan,)*3
    print("\n=== Paired bootstrap: Stage(combined) - benchmark (same patients) ===")
    for ref_name, ref in [("MELD-Na max", p_meld), ("FIB-4 max", p_fib)]:
        md,lo,hi=_paired_delta(y, p_stage, ref)
        sig="** significant" if (lo>0 or hi<0) else "ns (CI crosses 0)"
        print(f"  Stage - {ref_name:<12} = {md:+.3f} [95% CI {lo:+.3f}, {hi:+.3f}]  {sig}")

# ---- full Cox (hazard ratios + C-index) if lifelines available ----
if HAVE_LL:
    def cox(cols,label):
        cc=[c for c in cols if c in D.columns]; d=D[cc+["dur","event"]].dropna()
        if d.event.sum()<10 or len(d)<50: print(f"  [{label}] too few"); return
        for c in cc: d[c]=(d[c]-d[c].mean())/(d[c].std()+1e-9)
        cph=CoxPHFitter(penalizer=0.1).fit(d,"dur","event")
        ci=concordance_index(d["dur"],-cph.predict_partial_hazard(d),d["event"])
        print(f"\n[{label}] C-index={ci:.3f} (n={len(d)}, events={int(d.event.sum())})")
        h=cph.summary[["exp(coef)","exp(coef) lower 95%","exp(coef) upper 95%","p"]]; h.columns=["HR","lo","hi","p"]
        print(h.round(3).to_string())
    print("\n=== Cox proportional hazards (HR per 1 SD) ===")
    cox(["meld_max"],"MELD-Na"); cox(["fib4_max"],"FIB-4")
    cox(["peak_stage","stage_velocity","dwell_top","stage_entropy"],"Latent stage + dynamics")
    cox(["base_stage","fib4_base","meld_base"],"Baseline-only (clean)")
SF.to_csv(OUT/"progression_survival_features.csv", index=False)
print("\n✅ Saved:", OUT/"progression_survival_features.csv")


=== 1-year decompensation risk by baseline latent stage (Kaplan-Meier) ===
  baseline stage 0: n=1034  1-year risk = 0.7%
  baseline stage 1: n=352   1-year risk = 4.0%
  baseline stage 2: n=293   1-year risk = 3.4%

=== 1-year decompensation discrimination (n=1057, events=23) ===
  MELD-Na max      AUC=0.609 [0.471-0.739]
  FIB-4 max        AUC=0.627 [0.489-0.744]
  Peak stage       AUC=0.683 [0.572-0.774]
  Stage velocity   AUC=0.592 [0.422-0.742]
  Dwell top        AUC=0.654 [0.528-0.779]
  Stage (combined) AUC=0.662
  Stage+MELD+FIB-4 AUC=0.624

=== Paired bootstrap: Stage(combined) - benchmark (same patients) ===
  Stage - MELD-Na max  = +0.032 [95% CI -0.153, +0.193]  ns (CI crosses 0)
  Stage - FIB-4 max    = -0.005 [95% CI -0.197, +0.174]  ns (CI crosses 0)

=== Cox proportional hazards (HR per 1 SD) ===

[MELD-Na] C-index=0.642 (n=1228, events=77)
              HR     lo     hi      p
covariate                            
meld_max   1.179  1.034  1.343  0.014

[FIB-4] C-index=

In [8]:
# ============================================================
# STEP 3c — GROUPED K-FOLD OUT-OF-FOLD (leakage-free AND fully powered)
# Refit the HMM on train folds, score the held-out fold, pool all patients.
# Uses every event for evaluation while keeping train/test separation:
# fair like SF_test, powered like the full cohort. Needs `stage_features`
# (cell 9) and the imports/HAVE_LL from cell 12.
# ============================================================
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score, brier_score_loss

K_FOLDS = 5
_rng = np.random.default_rng(RANDOM_SEED)
_idx = _rng.permutation(len(seqs))
_folds = np.array_split(_idx, K_FOLDS)

_oof = []
for f in range(K_FOLDS):
    _te = set(_folds[f].tolist())
    _tr_seqs = [seqs[i] for i in range(len(seqs)) if i not in _te]
    _te_seqs = [seqs[i] for i in range(len(seqs)) if i in _te]
    _P = fit(_tr_seqs, K=K_STAGE, mode="baseline", n_iter=30, seed=RANDOM_SEED)
    _sev = _P["mu"] @ np.array([SEV_DIR[c] for c in OBS]); _o = np.argsort(_sev)
    _P = dict(_P, mu=_P["mu"][_o], var=_P["var"][_o],
              pi=_P["pi"][_o] / _P["pi"][_o].sum(), A=_P["A"][np.ix_(_o, _o)])
    _oof.append(stage_features(_P, _te_seqs))
SF_oof = pd.concat(_oof, ignore_index=True)
Do = SF_oof.copy(); Do["dur"] = Do["duration_days"].clip(lower=1)
print(f"OOF pooled: {len(Do):,} patients from {K_FOLDS} grouped folds")

def _km(dur, ev, t=365):
    dur = np.asarray(dur, float); ev = np.asarray(ev, int); s = 1.0
    for ut in np.unique(dur[ev == 1]):
        if ut > t: break
        d = int(((dur == ut) & (ev == 1)).sum()); r = int((dur >= ut).sum())
        if r > 0: s *= (1 - d / r)
    return 1 - s
print("\n1-year decompensation risk by baseline stage (OOF):")
for k in range(K_STAGE):
    m = Do["base_stage"].round() == k
    if m.sum() < 20: continue
    print(f"  stage {k}: n={int(m.sum()):<5} 1-yr risk = {_km(Do.loc[m,'dur'], Do.loc[m,'event'])*100:.1f}%")

_lab = np.where((Do.event == 1) & (Do.dur <= 365), 1, np.where(Do.dur >= 365, 0, -1))
_sel = _lab >= 0; yo = _lab[_sel]; Dq = Do[_sel].copy()
print(f"\n1-year discrimination (OOF): n={int(_sel.sum())}, events={int(yo.sum())}")

def _boot(yt, s, n=2000, seed=42):
    rng = np.random.default_rng(seed); a = []; yt = np.asarray(yt); s = np.asarray(s)
    for _ in range(n):
        i = rng.integers(0, len(yt), len(yt))
        if 1 < yt[i].sum() < len(i) - 1: a.append(roc_auc_score(yt[i], s[i]))
    return (np.median(a), np.percentile(a, 2.5), np.percentile(a, 97.5)) if a else (np.nan,) * 3
for nm, col in [("MELD-Na max", "meld_max"), ("FIB-4 max", "fib4_max"),
                ("Peak stage", "peak_stage"), ("Dwell top", "dwell_top")]:
    if col not in Dq.columns: continue
    m = Dq[col].notna().values
    if m.sum() < 20 or len(np.unique(yo[m])) < 2: continue
    md, lo, hi = _boot(yo[m], Dq[col].values[m]); print(f"  {nm:<14} AUC={md:.3f} [{lo:.3f}-{hi:.3f}]")

_sfeat = ["peak_stage", "stage_velocity", "dwell_top", "stage_entropy"]
_Xs = StandardScaler().fit_transform(SimpleImputer(strategy="median").fit_transform(Dq[_sfeat]))
p_stage = cross_val_predict(LogisticRegression(max_iter=2000, class_weight="balanced"),
                            _Xs, yo, cv=5, method="predict_proba")[:, 1]
print(f"  {'Stage (combined)':<14} AUC={roc_auc_score(yo, p_stage):.3f}")

def _pd(yt, a, b, n=2000, seed=42):
    m = ~(np.isnan(a) | np.isnan(b)); yt = np.asarray(yt)[m]; a = np.asarray(a)[m]; b = np.asarray(b)[m]
    rng = np.random.default_rng(seed); d = []
    for _ in range(n):
        i = rng.integers(0, len(yt), len(yt))
        if 1 < yt[i].sum() < len(i) - 1: d.append(roc_auc_score(yt[i], a[i]) - roc_auc_score(yt[i], b[i]))
    return (np.median(d), np.percentile(d, 2.5), np.percentile(d, 97.5)) if d else (np.nan,) * 3
print("\nPaired bootstrap Delta-AUC (OOF, same patients):")
for nm, ref in [("MELD-Na", Dq["meld_max"].values.astype(float)),
                ("FIB-4", Dq["fib4_max"].values.astype(float))]:
    md, lo, hi = _pd(yo, p_stage, ref)
    sig = "** significant" if (lo > 0 or hi < 0) else "ns (CI crosses 0)"
    print(f"  Stage(combined) - {nm:<8} = {md:+.3f} [95% CI {lo:+.3f}, {hi:+.3f}]  {sig}")

if HAVE_LL:
    def _ci(cols, label):
        cc = [c for c in cols if c in Do.columns]; d = Do[cc + ["dur", "event"]].dropna()
        for c in cc: d[c] = (d[c] - d[c].mean()) / (d[c].std() + 1e-9)
        cph = CoxPHFitter(penalizer=0.1).fit(d, "dur", "event")
        ci = concordance_index(d["dur"], -cph.predict_partial_hazard(d), d["event"])
        print(f"  [{label}] C-index={ci:.3f} (events={int(d.event.sum())})")
    print("\nCox C-index (OOF):")
    _ci(["meld_max"], "MELD-Na"); _ci(["fib4_max"], "FIB-4"); _ci(_sfeat, "Latent stage + dynamics")

p_cal = cross_val_predict(LogisticRegression(max_iter=2000), _Xs, yo, cv=5, method="predict_proba")[:, 1]
print(f"\n1-year calibration (OOF, Brier={brier_score_loss(yo, p_cal):.3f})")
SF_oof.to_csv(OUT / "progression_oof_features.csv", index=False)
print("Saved:", OUT / "progression_oof_features.csv")


OOF pooled: 5,597 patients from 5 grouped folds

1-year decompensation risk by baseline stage (OOF):
  stage 0: n=3365  1-yr risk = 0.6%
  stage 1: n=1280  1-yr risk = 3.2%
  stage 2: n=952   1-yr risk = 4.9%

1-year discrimination (OOF): n=3611, events=79
  MELD-Na max    AUC=0.675 [0.619-0.731]
  FIB-4 max      AUC=0.616 [0.544-0.685]
  Peak stage     AUC=0.737 [0.677-0.788]
  Dwell top      AUC=0.718 [0.653-0.780]
  Stage (combined) AUC=0.790

Paired bootstrap Delta-AUC (OOF, same patients):
  Stage(combined) - MELD-Na  = +0.114 [95% CI +0.049, +0.180]  ** significant
  Stage(combined) - FIB-4    = +0.171 [95% CI +0.097, +0.242]  ** significant

Cox C-index (OOF):
  [MELD-Na] C-index=0.654 (events=269)
  [FIB-4] C-index=0.665 (events=280)
  [Latent stage + dynamics] C-index=0.737 (events=301)

1-year calibration (OOF, Brier=0.021)
Saved: results/progression_output/progression_oof_features.csv


In [9]:
# ============================================================
# STEP 3b — 1-year decompensation calibration (5-fold OOF) + Brier
# ============================================================
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import brier_score_loss

_cc = [c for c in ["peak_stage", "stage_velocity", "dwell_top", "stage_entropy",
                   "meld_max", "fib4_max"] if c in Dd.columns]
_Xc = StandardScaler().fit_transform(SimpleImputer(strategy="median").fit_transform(Dd[_cc]))
# plain logistic (NO class_weight) so predicted probabilities are calibratable
_p = cross_val_predict(LogisticRegression(max_iter=2000), _Xc, y, cv=5,
                       method="predict_proba")[:, 1]
_brier = brier_score_loss(y, _p)
print(f"=== 1-year decompensation calibration (5-fold OOF, Brier = {_brier:.3f}) ===")
_edges = np.quantile(_p, np.linspace(0, 1, 6)); _edges[0] -= 1e-9
_bin = np.digitize(_p, _edges[1:-1])
for b in range(5):
    m = _bin == b
    if m.sum() < 10: continue
    print(f"  bin {b}: n={int(m.sum()):<4} predicted={_p[m].mean():.3f}  observed={y[m].mean():.3f}")

=== 1-year decompensation calibration (5-fold OOF, Brier = 0.021) ===
  bin 0: n=212  predicted=0.006  observed=0.024
  bin 1: n=211  predicted=0.011  observed=0.005
  bin 2: n=211  predicted=0.012  observed=0.009
  bin 3: n=211  predicted=0.017  observed=0.024
  bin 4: n=212  predicted=0.059  observed=0.047


## Step 4 — Explainability (normalized LLR vs SHAP)

In [10]:
# ============================================================
# EXPLAINABILITY — LLR (raw + variance-normalized) vs SHAP
# ============================================================
from scipy.stats import spearmanr
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
try:
    import shap as _shap; _HS = True
except Exception:
    _HS = False; print("shap not installed -> exact-linear fallback")

_P = params; _S = seqs; _OBS = list(OBS); _K = K_STAGE
_mu, _var = _P["mu"], _P["var"]

def _llr_feat(o, tgt, ref=0, unit=False):
    out = {}
    for j, f in enumerate(_OBS):
        if np.isnan(o[j]): continue
        if unit:   # unit-variance / LDA-style: scale-invariant on standardized obs
            out[f] = (_mu[tgt,j]-_mu[ref,j]) * (o[j]-0.5*(_mu[tgt,j]+_mu[ref,j]))
        else:      # original variance-weighted LLR
            lt = -0.5*(np.log(2*np.pi*_var[tgt,j]) + (o[j]-_mu[tgt,j])**2/_var[tgt,j])
            lr = -0.5*(np.log(2*np.pi*_var[ref,j]) + (o[j]-_mu[ref,j])**2/_var[ref,j])
            out[f] = lt - lr
    return out

_raw = {f: [] for f in _OBS}; _nrm = {f: [] for f in _OBS}; _X = []; _Y = []
for s in _S:
    g, st, _ = posterior_stage(_P, s["O"]); hard = np.argmax(g, axis=1)
    for t in range(len(st)):
        _X.append(s["O"][t]); _Y.append(int(hard[t] == _K-1))
        if hard[t] == _K-1:
            for f, v in _llr_feat(s["O"][t], _K-1, 0, False).items(): _raw[f].append(abs(v))
            for f, v in _llr_feat(s["O"][t], _K-1, 0, True ).items(): _nrm[f].append(abs(v))
Xv = pd.DataFrame(_X, columns=_OBS); Xv = Xv.fillna(Xv.median()); Yv = np.array(_Y)
print(f"visit-level: {len(Xv):,} visits | advanced (top stage) = {int(Yv.sum()):,} ({Yv.mean()*100:.1f}%)")

llr_raw  = pd.Series({f:(np.mean(v) if v else 0.0) for f,v in _raw.items()})
llr_norm = pd.Series({f:(np.mean(v) if v else 0.0) for f,v in _nrm.items()})
_gbm = GradientBoostingClassifier(max_depth=3, n_estimators=200, random_state=0).fit(Xv, Yv)
if _HS:
    _r = _shap.TreeExplainer(_gbm).shap_values(Xv)
    if isinstance(_r, list): _r = _r[-1]
    _r = np.asarray(_r)
    if _r.ndim == 3: _r = _r[:, :, -1]
    shap_imp = pd.Series(np.abs(_r).mean(0), index=_OBS)
else:
    _lin = LogisticRegression(max_iter=2000).fit(Xv, Yv)
    shap_imp = pd.Series(np.abs(_lin.coef_[0])*Xv.std().values, index=_OBS)

cmp = pd.DataFrame({"LLR_raw":llr_raw, "LLR_norm":llr_norm, "SHAP":shap_imp})
for c in ["LLR_raw","LLR_norm","SHAP"]: cmp[c+"_rk"] = cmp[c].rank(ascending=False)
cmp = cmp.sort_values("SHAP", ascending=False)
rho_raw  = spearmanr(cmp["LLR_raw_rk"],  cmp["SHAP_rk"]).correlation
rho_norm = spearmanr(cmp["LLR_norm_rk"], cmp["SHAP_rk"]).correlation
print("\n=== Feature importance: raw-LLR vs norm-LLR vs SHAP (advanced stage) ===")
print(cmp[["LLR_raw","LLR_norm","SHAP","LLR_raw_rk","LLR_norm_rk","SHAP_rk"]].round(3).to_string())
print(f"\nSpearman  SHAP vs raw-LLR  = {rho_raw:+.3f}")
print(f"Spearman  SHAP vs norm-LLR = {rho_norm:+.3f}")
print("  If norm-LLR agrees with SHAP much better, the raw-LLR ranking was a variance artifact.")


visit-level: 23,636 visits | advanced (top stage) = 4,479 (18.9%)

=== Feature importance: raw-LLR vs norm-LLR vs SHAP (advanced stage) ===
            LLR_raw  LLR_norm   SHAP  LLR_raw_rk  LLR_norm_rk  SHAP_rk
INR          16.339     5.459  0.847         1.0          1.0      1.0
Bilirubin     6.771     3.384  0.692         2.0          2.0      2.0
AST           1.930     1.279  0.504         4.0          3.0      3.0
ALT           1.544     0.996  0.423         5.0          4.0      4.0
Creatinine    2.520     0.640  0.366         3.0          6.0      5.0
Platelets     1.106     0.098  0.274         6.0          9.0      6.0
Sodium        0.635     0.457  0.264         8.0          7.0      7.0
Hemoglobin    0.555     0.349  0.174         9.0          8.0      8.0
Albumin       1.105     0.848  0.142         7.0          5.0      9.0
WBC           0.387     0.069  0.093        10.0         10.0     10.0

Spearman  SHAP vs raw-LLR  = +0.927
Spearman  SHAP vs norm-LLR = +0.842
  If n

## Step 5 — Visualization
Figures saved to `results/progression_output/figures/`.

In [11]:
# ============================================================
# VIZ setup + shared palette + V1: KM by baseline stage
# ============================================================
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
FIG = OUT / "figures"; FIG.mkdir(parents=True, exist_ok=True)

# unified blue-primary scheme (matches PEMDT-AKI and respiratory)
RED, BLUE, GREEN, GREY, AMBER = "#e63946", "#1f77b4", "#2ca02c", "#9aa0a6", "#f4a261"
STAGE_COLORS = [BLUE, GREEN, AMBER, RED]  # stages 0->3 (blue, green, amber, red only worst)
C_STAGE = BLUE    # latent stage / PEMDT model (primary)
C_MELD  = GREEN   # MELD-Na
C_FIB   = AMBER   # FIB-4
C_SHAP  = GREEN   # SHAP series
C_REF   = GREY    # reference / chance / perfect line (neutral, not a data color)

# --- evaluation figures use the OUT-OF-FOLD pooled set if available (powered + leakage-free) ---
Dfig  = Do if "Do" in globals() else D
Ddfig = Dq if "Dq" in globals() else Dd
yfig  = yo if "yo" in globals() else y
print("figures using:", "OOF pooled set" if "Do" in globals() else "SF_test (run the OOF cell for pooled figures)")

def km_curve(dur, ev, tmax=730):
    dur = np.asarray(dur, float); ev = np.asarray(ev, int)
    ts = np.unique(dur[ev == 1]); ts = ts[ts <= tmax]
    surv = 1.0; xs = [0.0]; ys = [0.0]
    for ut in ts:
        d = int(((dur == ut) & (ev == 1)).sum()); r = int((dur >= ut).sum())
        if r > 0: surv *= (1 - d / r)
        xs.append(ut); ys.append(1 - surv)
    xs.append(tmax); ys.append(ys[-1])
    return np.array(xs), np.array(ys) * 100

plt.figure(figsize=(7, 5))
for k in range(K_STAGE):
    m = Dfig["base_stage"].round() == k
    if m.sum() < 20: continue
    x, yv = km_curve(Dfig.loc[m, "dur"], Dfig.loc[m, "event"])
    r1 = yv[np.searchsorted(x, 365, "right") - 1]
    plt.step(x / 365, yv, where="post", lw=2.2, color=STAGE_COLORS[k % 4],
             label=f"Baseline stage {k} (n={int(m.sum())}, 1-yr {r1:.1f}%)")
plt.xlabel("Years from baseline"); plt.ylabel("Cumulative decompensation risk (%)")
plt.title("Decompensation by baseline latent stage")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig(FIG / "km_by_stage.png", dpi=150); plt.show()


figures using: OOF pooled set


In [12]:
# ============================================================
# V2: ROC — latent stage vs MELD-Na vs FIB-4 (1-year discrimination)
# ============================================================
from sklearn.metrics import roc_curve, roc_auc_score
plt.figure(figsize=(6, 6))
for name, col, c in [("Latent peak stage", "peak_stage", C_STAGE),
                     ("MELD-Na", "meld_max", C_MELD),
                     ("FIB-4", "fib4_max", C_FIB)]:
    if col not in Ddfig.columns: continue
    m = Ddfig[col].notna().values
    if len(np.unique(yfig[m])) < 2: continue
    s = Ddfig[col].values[m]
    fpr, tpr, _ = roc_curve(yfig[m], s)
    plt.plot(fpr, tpr, lw=2, color=c, label=f"{name} (AUC={roc_auc_score(yfig[m], s):.3f})")
plt.plot([0, 1], [0, 1], "--", color=C_REF, lw=1)
plt.xlabel("False-positive rate"); plt.ylabel("True-positive rate")
plt.title("1-year decompensation discrimination")
plt.legend(loc="lower right"); plt.tight_layout()
plt.savefig(FIG / "roc_stage_vs_scores.png", dpi=150); plt.show()


In [13]:
# ============================================================
# V3: Calibration curve (predicted vs observed 1-year risk)
# ============================================================
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import brier_score_loss
_cc = [c for c in ["peak_stage", "stage_velocity", "dwell_top", "stage_entropy",
                   "meld_max", "fib4_max"] if c in Ddfig.columns]
_X = StandardScaler().fit_transform(SimpleImputer(strategy="median").fit_transform(Ddfig[_cc]))
p = cross_val_predict(LogisticRegression(max_iter=2000), _X, yfig, cv=5, method="predict_proba")[:, 1]
edges = np.quantile(p, np.linspace(0, 1, 6)); edges[0] -= 1e-9; b = np.digitize(p, edges[1:-1])
xs = [p[b == i].mean() for i in range(5) if (b == i).sum()]
ys = [yfig[b == i].mean() for i in range(5) if (b == i).sum()]
hi = max(max(xs), max(ys)) * 1.1
plt.figure(figsize=(6, 6))
plt.plot([0, hi], [0, hi], "--", color=C_REF, label="perfect")
plt.plot(xs, ys, "o-", color=C_STAGE, lw=2, label="model")
plt.xlabel("Predicted 1-year risk"); plt.ylabel("Observed 1-year risk")
plt.title(f"1-year risk calibration (Brier={brier_score_loss(yfig, p):.3f})")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig(FIG / "calibration_1yr.png", dpi=150); plt.show()


In [14]:
# ============================================================
# V4: Cox hazard-ratio forest plot (colored by feature group)
# ============================================================
if HAVE_LL:
    def _hr(cols):
        cc = [c for c in cols if c in Dfig.columns]; d = Dfig[cc + ["dur", "event"]].dropna()
        for c in cc: d[c] = (d[c] - d[c].mean()) / (d[c].std() + 1e-9)
        cph = CoxPHFitter(penalizer=0.1).fit(d, "dur", "event"); h = cph.summary
        return [(c, h.loc[c, "exp(coef)"], h.loc[c, "exp(coef) lower 95%"],
                 h.loc[c, "exp(coef) upper 95%"]) for c in cc]
    rows = _hr(["peak_stage", "stage_velocity", "dwell_top", "stage_entropy"]) + \
           _hr(["meld_max", "fib4_max"])
    _cmap = {"meld_max": C_MELD, "fib4_max": C_FIB}
    lab = [r[0] for r in rows]; hr = np.array([r[1] for r in rows])
    lo = np.array([r[2] for r in rows]); hiv = np.array([r[3] for r in rows])
    yv = np.arange(len(rows))[::-1]
    plt.figure(figsize=(7, 4))
    for i in range(len(rows)):
        plt.errorbar(hr[i], yv[i], xerr=[[hr[i] - lo[i]], [hiv[i] - hr[i]]],
                     fmt="o", color=_cmap.get(lab[i], C_STAGE), capsize=3)
    plt.axvline(1, color=C_REF, ls="--"); plt.yticks(yv, lab)
    plt.xlabel("Hazard ratio per 1 SD (95% CI)")
    plt.title("Cox proportional hazards"); plt.tight_layout()
    plt.savefig(FIG / "cox_forest.png", dpi=150); plt.show()
else:
    print("lifelines not available; skipping forest plot")


In [15]:
# ============================================================
# V5: Example patient latent-stage trajectory (digital-twin view)
# ============================================================
_prog = set(surv.loc[surv.event == 1, "subject_id"])
_cand = [s for s in seqs if len(s["O"]) >= 5 and s["pid"] in _prog] or \
        [s for s in seqs if len(s["O"]) >= 5]
s = max(_cand, key=lambda z: len(z["O"]))
g, st, _ = posterior_stage(params, s["O"])
days = (pd.to_datetime(s["admit"]).astype("int64") / 1e9 / 86400); days = days - days[0]
plt.figure(figsize=(9, 4))
plt.stackplot(days, *[g[:, k] for k in range(K_STAGE)],
              labels=[f"stage {k}" for k in range(K_STAGE)],
              colors=STAGE_COLORS[:K_STAGE], alpha=.85)
plt.plot(days, st / (K_STAGE - 1), "k--", lw=1.3, label="hard stage / max")
plt.xlabel("Days from first visit"); plt.ylabel("Posterior belief"); plt.ylim(0, 1)
plt.title(f"Latent-stage trajectory (patient {s['pid']})")
plt.legend(ncol=3, fontsize=8, loc="upper left"); plt.tight_layout()
plt.savefig(FIG / "patient_trajectory.png", dpi=150); plt.show()


In [16]:
# ============================================================
# V6: LLR vs SHAP feature importance (needs `cmp` from explainability cell)
# ============================================================
try:
    _c = cmp.copy()
    _lcol = "LLR_norm" if "LLR_norm" in _c.columns else ("LLR" if "LLR" in _c.columns else "LLR_raw")
    _c = _c.sort_values("SHAP", ascending=False)
    def _nz(sr): return sr / (sr.max() if sr.max() > 0 else 1.0)
    xv = np.arange(len(_c)); w = 0.4
    plt.figure(figsize=(9, 4))
    plt.bar(xv - w / 2, _nz(_c[_lcol]), w, label="LLR (normalized)", color=C_STAGE)
    plt.bar(xv + w / 2, _nz(_c["SHAP"]), w, label="SHAP", color=C_SHAP)
    plt.xticks(xv, _c.index, rotation=45, ha="right"); plt.ylabel("Relative importance")
    plt.title("Feature importance: LLR vs SHAP (advanced stage)")
    plt.legend(); plt.tight_layout()
    plt.savefig(FIG / "llr_vs_shap.png", dpi=150); plt.show()
except NameError:
    print("Run the explainability cell (defines `cmp`) first.")
